In [1]:
!apt-get update -qq
!apt-get install -y swi-prolog
!pip install pyswip

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  autopoint debhelper debugedit dh-autoreconf dh-strip-nondeterminism dwz
  gettext gettext-base intltool-debian libarchive-cpio-perl
  libarchive-zip-perl libdebhelper-perl libfile-stripnondeterminism-perl
  libmail-sendmail-perl libossp-uuid16 libsub-override-perl
  libsys-hostname-long-perl libtool po-debconf swi-prolog-core
  swi-prolog-core-packages swi-prolog-doc swi-prolog-nox swi-prolog-x
Suggested packages:
  dh-make gettext-doc libasprintf-dev libgettextpo-dev uuid libtool-doc
  gcj-jdk libmail-box-perl elpa-ediprolog swi-prolog-java swi-prolog-odbc
  swi-prolog-bdb
The following NEW packages will be installed:
  autopoint debhelper debugedit 

In [2]:
!swipl --version

SWI-Prolog version 8.4.2 for x86_64-linux


In [3]:
from pathlib import Path

BASE_DIR = Path("/content/CineLog")
BASE_DIR.mkdir(exist_ok=True)

print(BASE_DIR)

/content/CineLog


In [4]:
import os

print(os.listdir("/content/CineLog"))

['tmdb_final_preprocessed.csv', 'tmdb_lda_topic_mapping.csv']


csv 구조 확인하기

In [5]:
import pandas as pd

movie_path = "/content/CineLog/tmdb_final_preprocessed.csv"
topic_path = "/content/CineLog/tmdb_lda_topic_mapping.csv"

movies = pd.read_csv(movie_path)
topics = pd.read_csv(topic_path)

print("movies shape:", movies.shape)
print("topics shape:", topics.shape)

print("\n[Movie Columns]")
print(movies.columns.tolist())

print("\n[Topic Columns]")
print(topics.columns.tolist())

movies.head(3)

movies shape: (4809, 11)
topics shape: (4809, 12)

[Movie Columns]
['id', 'title', 'release_date', 'vote_average', 'runtime', 'popularity', 'genres_clean', 'cast_clean', 'director_clean', 'overview', 'keywords_tfidf']

[Topic Columns]
['id', 'title', 'topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5', 'topic_6', 'topic_7', 'topic_8', 'topic_9']


,id,title,release_date,vote_average,runtime,popularity,genres_clean,cast_clean,director_clean,overview,keywords_tfidf
0,19995,Avatar,2009-12-10,7.2,162.0,150.437577,"['action', 'adventure', 'fantasy', 'science_fi...","['sam_worthington', 'zoe_saldana', 'sigourney_...",['james_cameron'],"In the 22nd century, a paraplegic Marine is di...","['marine', 'alien', 'orders', 'futuristic', 'm..."
1,285,Pirates of the Caribbean: At World's End,2007-05-19,6.9,169.0,139.082615,"['adventure', 'fantasy', 'action']","['johnny_depp', 'orlando_bloom', 'keira_knight...",['gore_verbinski'],"Captain Barbossa, long believed to be dead, ha...","['headed', 'traitor', 'quite', 'elizabeth', 'a..."
2,206647,Spectre,2015-10-26,6.3,148.0,107.376788,"['action', 'adventure', 'crime']","['daniel_craig', 'christoph_waltz', 'léa_seydo...",['sam_mendes'],A cryptic message from Bond’s past sends him o...,"['bond', 'british_secret_service', 'reveal', '..."


# Rule Mapping으로 Prolog fact 형식으로 변환

In [6]:
import pandas as pd
import ast
import re
from pathlib import Path

BASE_DIR = Path("/content/CineLog")

MOVIE_CSV = BASE_DIR / "tmdb_final_preprocessed.csv"
TOPIC_CSV = BASE_DIR / "tmdb_lda_topic_mapping.csv"

FACT_OUTPUT = BASE_DIR / "movie_facts.pl"
KG_OUTPUT = BASE_DIR / "kg_edges.csv"

# Topic_Dictionary.docx의 해석을 Prolog atom으로 변환한 매핑
TOPIC_MAP = {
    "topic_0": "family_generational_drama",
    "topic_1": "war_epic_period",
    "topic_2": "human_journey_true_story",
    "topic_3": "crime_thriller_revenge",
    "topic_4": "romantic_comedy_family",
    "topic_5": "teen_youth_indie",
    "topic_6": "mystery_occult_adventure",
    "topic_7": "campus_subculture_social",
    "topic_8": "sf_blockbuster_dystopia",
    "topic_9": "spy_investigation_action",
}

# 각 영화별로 상위 토픽 3개만 저장
TOPIC_TOP_N = 3

# 토픽 점수가 너무 낮으면 저장하지 않음
TOPIC_THRESHOLD = 0.15


def movie_atom(movie_id):
    """
    영화 제목은 중복될 수 있으므로 title이 아니라 TMDB id를 사용.
    예: 19995 -> m19995
    """
    return f"m{int(movie_id)}"


def safe_atom(value):
    """
    Prolog atom으로 안전하게 변환.
    예: 'Science Fiction' -> science_fiction
    예: 'Sam Worthington' -> sam_worthington
    """
    if pd.isna(value):
        return None

    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")

    if not value:
        return None

    if value[0].isdigit():
        value = "n_" + value

    return value


def quote_string(value):
    """
    Prolog에서 제목을 안전하게 저장하기 위한 quoted atom 처리.
    예: Avatar -> 'Avatar'
    """
    if pd.isna(value):
        return "''"

    value = str(value)
    value = value.replace("\\", "\\\\").replace("'", "\\'")
    return f"'{value}'"


def parse_list_column(value):
    """
    "['action', 'adventure']" 형태의 문자열을 Python list로 변환.
    """
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return parsed
    except Exception:
        pass

    return []


def clean_year(date_value):
    """
    release_date에서 연도만 추출.
    예: 2009-12-10 -> 2009
    """
    if pd.isna(date_value):
        return None

    dt = pd.to_datetime(date_value, errors="coerce")
    if pd.isna(dt):
        return None

    return int(dt.year)


def write_fact(f, fact):
    f.write(f"{fact}\n")


def generate_movie_facts():
    movies = pd.read_csv(MOVIE_CSV)
    topics = pd.read_csv(TOPIC_CSV)

    # id 기준 병합
    df = movies.merge(
        topics.drop(columns=["title"], errors="ignore"),
        on="id",
        how="left"
    )

    kg_edges = []

    with open(FACT_OUTPUT, "w", encoding="utf-8") as f:
        f.write("% Auto-generated Prolog facts from TMDB dataset\n")
        f.write("% STEP2: Knowledge Representation\n\n")

        for _, row in df.iterrows():
            m = movie_atom(row["id"])

            # 기본 영화 정보
            write_fact(f, f"movie({m}).")
            write_fact(f, f"title({m}, {quote_string(row.get('title'))}).")

            year = clean_year(row.get("release_date"))
            if year is not None:
                write_fact(f, f"release_year({m}, {year}).")

            if "vote_average" in row and not pd.isna(row.get("vote_average")):
                write_fact(f, f"rating({m}, {float(row['vote_average']):.2f}).")

            if "runtime" in row and not pd.isna(row.get("runtime")):
                write_fact(f, f"runtime({m}, {int(float(row['runtime']))}).")

            if "popularity" in row and not pd.isna(row.get("popularity")):
                write_fact(f, f"popularity({m}, {float(row['popularity']):.6f}).")

            # 장르 fact
            for genre in parse_list_column(row.get("genres_clean")):
                g = safe_atom(genre)
                if g:
                    write_fact(f, f"genre({m}, {g}).")
                    kg_edges.append([m, "has_genre", g])

            # 배우 fact: 상위 5명만 사용
            for actor in parse_list_column(row.get("cast_clean"))[:5]:
                a = safe_atom(actor)
                if a:
                    write_fact(f, f"actor({m}, {a}).")
                    kg_edges.append([m, "has_actor", a])

            # 감독 fact
            for director in parse_list_column(row.get("director_clean")):
                d = safe_atom(director)
                if d:
                    write_fact(f, f"director({m}, {d}).")
                    kg_edges.append([m, "has_director", d])

            # TF-IDF keyword fact: 상위 10개만 사용
            for keyword in parse_list_column(row.get("keywords_tfidf"))[:10]:
                k = safe_atom(keyword)
                if k:
                    write_fact(f, f"keyword({m}, {k}).")
                    kg_edges.append([m, "has_keyword", k])

            # LDA topic fact
            topic_scores = []
            for topic_col, topic_atom in TOPIC_MAP.items():
                if topic_col in row and not pd.isna(row[topic_col]):
                    topic_scores.append((topic_atom, float(row[topic_col])))

            topic_scores = sorted(topic_scores, key=lambda x: x[1], reverse=True)

            for topic_atom, score in topic_scores[:TOPIC_TOP_N]:
                if score >= TOPIC_THRESHOLD:
                    write_fact(f, f"has_topic({m}, {topic_atom}, {score:.6f}).")
                    kg_edges.append([m, "has_topic", topic_atom])

            f.write("\n")

    pd.DataFrame(
        kg_edges,
        columns=["source", "relation", "target"]
    ).to_csv(KG_OUTPUT, index=False, encoding="utf-8-sig")

    print(f"Saved: {FACT_OUTPUT}")
    print(f"Saved: {KG_OUTPUT}")
    print(f"Total movies: {len(df)}")
    print(f"Total KG edges: {len(kg_edges)}")


generate_movie_facts()

Saved: /content/CineLog/movie_facts.pl
Saved: /content/CineLog/kg_edges.csv
Total movies: 4821
Total KG edges: 65025


In [7]:
print(os.listdir("/content/CineLog"))

['tmdb_final_preprocessed.csv', 'tmdb_lda_topic_mapping.csv', 'movie_facts.pl', 'kg_edges.csv']


In [8]:
with open("/content/CineLog/movie_facts.pl", "r", encoding="utf-8") as f:
    for i in range(40):
        print(f.readline().rstrip())

% Auto-generated Prolog facts from TMDB dataset
% STEP2: Knowledge Representation

movie(m19995).
title(m19995, 'Avatar').
release_year(m19995, 2009).
rating(m19995, 7.20).
runtime(m19995, 162).
popularity(m19995, 150.437577).
genre(m19995, action).
genre(m19995, adventure).
genre(m19995, fantasy).
genre(m19995, science_fiction).
actor(m19995, sam_worthington).
actor(m19995, zoe_saldana).
actor(m19995, sigourney_weaver).
director(m19995, james_cameron).
keyword(m19995, marine).
keyword(m19995, alien).
keyword(m19995, orders).
keyword(m19995, futuristic).
keyword(m19995, moon).
has_topic(m19995, sf_blockbuster_dystopia, 0.533787).
has_topic(m19995, human_journey_true_story, 0.264766).
has_topic(m19995, war_epic_period, 0.168111).

movie(m285).
title(m285, 'Pirates of the Caribbean: At World\'s End').
release_year(m285, 2007).
rating(m285, 6.90).
runtime(m285, 169).
popularity(m285, 139.082615).
genre(m285, adventure).
genre(m285, fantasy).
genre(m285, action).
actor(m285, johnny_depp).


**PySWIP으로 Prolog fact 로드 테스트**

In [9]:
from pyswip import Prolog

prolog = Prolog()
prolog.consult("/content/CineLog/movie_facts.pl")

print("movie_facts.pl loaded successfully.")

movie_facts.pl loaded successfully.


In [10]:
list(prolog.query("movie(M), title(M, T)", maxresult=5))

[{'M': 'm19995', 'T': 'Avatar'},
 {'M': 'm285', 'T': "Pirates of the Caribbean: At World's End"},
 {'M': 'm206647', 'T': 'Spectre'},
 {'M': 'm49026', 'T': 'The Dark Knight Rises'},
 {'M': 'm49529', 'T': 'John Carter'}]

In [11]:
list(prolog.query("title(m19995, T)"))

[{'T': 'Avatar'}]

In [12]:
list(prolog.query("genre(m19995, G)"))

[{'G': 'action'},
 {'G': 'adventure'},
 {'G': 'fantasy'},
 {'G': 'science_fiction'}]

In [13]:
list(prolog.query("has_topic(m19995, Topic, Score)"))

[{'Topic': 'sf_blockbuster_dystopia', 'Score': 0.533787},
 {'Topic': 'human_journey_true_story', 'Score': 0.264766},
 {'Topic': 'war_epic_period', 'Score': 0.168111}]

# Semantic Net 구축

isa/2 관계를 넣어서 Prolog가 상위 개념까지 추론할 수 있게 함

In [14]:
from pathlib import Path

BASE_DIR = Path("/content/CineLog")
ONTOLOGY_OUTPUT = BASE_DIR / "ontology_rules.pl"

ontology_text = r"""
% ============================================================
% CineLog Ontology Rules
% STEP2: Multi-label Semantic Net
% ============================================================

% ------------------------------------------------------------
% 1. Genre-level semantic hierarchy
% ------------------------------------------------------------
% Genre is treated as a surface-level category.
% A movie can still belong to other semantic categories through topics/keywords.

isa(action, action_genre).
isa(adventure, action_genre).

isa(comedy, light_genre).
isa(romance, light_genre).
isa(family, light_genre).
isa(animation, light_genre).

isa(drama, narrative_genre).
isa(history, narrative_genre).
isa(documentary, narrative_genre).

isa(crime, dark_genre).
isa(thriller, dark_genre).
isa(horror, dark_genre).
isa(mystery, dark_genre).

isa(science_fiction, speculative_genre).
isa(fantasy, speculative_genre).

isa(war, conflict_genre).
isa(western, conflict_genre).

% Higher-level genre abstractions
isa(action_genre, entertainment).
isa(light_genre, entertainment).
isa(narrative_genre, story_based).
isa(dark_genre, dark_story).
isa(speculative_genre, speculative_story).
isa(conflict_genre, conflict_story).


% ------------------------------------------------------------
% 2. LDA topic-level semantic hierarchy
% ------------------------------------------------------------
% Topic names are based on Topic_Dictionary.docx interpretation.

isa(family_generational_drama, human_drama).
isa(human_journey_true_story, human_drama).
isa(romantic_comedy_family, human_drama).
isa(teen_youth_indie, human_drama).

isa(war_epic_period, historical_conflict).
isa(historical_conflict, conflict_story).

isa(crime_thriller_revenge, dark_story).
isa(mystery_occult_adventure, dark_story).
isa(spy_investigation_action, action_thriller).
isa(action_thriller, dark_story).

isa(sf_blockbuster_dystopia, speculative_story).

isa(campus_subculture_social, social_issue).
isa(social_issue, human_drama).


% ------------------------------------------------------------
% 3. Keyword-level semantic hierarchy
% ------------------------------------------------------------
% Keywords provide fine-grained narrative clues.

isa(revenge, dark_story).
isa(murder, dark_story).
isa(killer, dark_story).
isa(violence, dark_story).
isa(crime, dark_story).
isa(police, investigation_story).
isa(detective, investigation_story).
isa(fbi, investigation_story).
isa(cia, investigation_story).
isa(investigation, investigation_story).
isa(undercover, investigation_story).
isa(terrorist, conflict_story).

isa(alien, speculative_story).
isa(space, speculative_story).
isa(planet, speculative_story).
isa(future, speculative_story).
isa(dystopia, speculative_story).
isa(robot, speculative_story).
isa(time_travel, speculative_story).

isa(family, human_drama).
isa(father, human_drama).
isa(mother, human_drama).
isa(son, human_drama).
isa(daughter, human_drama).
isa(love, human_drama).
isa(friendship, human_drama).
isa(coming_of_age, human_drama).

isa(school, youth_story).
isa(college, youth_story).
isa(teenager, youth_story).
isa(high_school, youth_story).
isa(youth_story, human_drama).

isa(war, conflict_story).
isa(battle, conflict_story).
isa(soldier, conflict_story).
isa(world_war_ii, conflict_story).


% ------------------------------------------------------------
% 4. Transitive inheritance
% ------------------------------------------------------------
% X is a sub-concept of Y either directly or through intermediate concepts.

isa_transitive(X, Y) :-
    isa(X, Y).

isa_transitive(X, Z) :-
    isa(X, Y),
    isa_transitive(Y, Z).


% ------------------------------------------------------------
% 5. Genre reasoning
% ------------------------------------------------------------

is_genre(Movie, Genre) :-
    genre(Movie, Genre).

is_genre(Movie, UpperGenre) :-
    genre(Movie, Genre),
    isa_transitive(Genre, UpperGenre).


% ------------------------------------------------------------
% 6. Topic reasoning
% ------------------------------------------------------------

topic_threshold(0.15).
strong_topic_threshold(0.30).

is_topic(Movie, Topic) :-
    topic_threshold(T),
    has_topic(Movie, Topic, Score),
    Score >= T.

is_topic(Movie, UpperTopic) :-
    topic_threshold(T),
    has_topic(Movie, Topic, Score),
    Score >= T,
    isa_transitive(Topic, UpperTopic).

strong_topic(Movie, Topic) :-
    strong_topic_threshold(T),
    has_topic(Movie, Topic, Score),
    Score >= T.


% ------------------------------------------------------------
% 7. Keyword reasoning
% ------------------------------------------------------------

is_keyword(Movie, Keyword) :-
    keyword(Movie, Keyword).

is_keyword(Movie, UpperKeyword) :-
    keyword(Movie, Keyword),
    isa_transitive(Keyword, UpperKeyword).


% ------------------------------------------------------------
% 8. Integrated semantic category reasoning
% ------------------------------------------------------------
% A movie can belong to a semantic category through:
% 1) genre
% 2) LDA topic
% 3) keyword
% This is the core of the multi-label Semantic Net.

has_semantic_category(Movie, Category) :-
    is_genre(Movie, Category).

has_semantic_category(Movie, Category) :-
    is_topic(Movie, Category).

has_semantic_category(Movie, Category) :-
    is_keyword(Movie, Category).


% ------------------------------------------------------------
% 9. Basic condition rules
% ------------------------------------------------------------

high_rated(Movie) :-
    rating(Movie, R),
    R >= 8.0.

good_rated(Movie) :-
    rating(Movie, R),
    R >= 7.0.

recent_movie(Movie) :-
    release_year(Movie, Y),
    Y >= 2015.

classic_movie(Movie) :-
    release_year(Movie, Y),
    Y < 2000.

short_movie(Movie) :-
    runtime(Movie, T),
    T =< 100.

long_movie(Movie) :-
    runtime(Movie, T),
    T >= 150.

popular_movie(Movie) :-
    popularity(Movie, P),
    P >= 50.


% ------------------------------------------------------------
% 10. Prototype recommendation rules
% ------------------------------------------------------------

recommend_by_semantic_category(Movie, Category, MinRating) :-
    movie(Movie),
    rating(Movie, R),
    R >= MinRating,
    has_semantic_category(Movie, Category).

recommend_dark_action(Movie) :-
    movie(Movie),
    good_rated(Movie),
    is_genre(Movie, action_genre),
    has_semantic_category(Movie, dark_story).

recommend_philosophical_sf(Movie) :-
    movie(Movie),
    good_rated(Movie),
    has_semantic_category(Movie, speculative_story),
    has_semantic_category(Movie, human_drama).

recommend_light_fun(Movie) :-
    movie(Movie),
    good_rated(Movie),
    popular_movie(Movie),
    has_semantic_category(Movie, entertainment).

recommend_emotional_movie(Movie) :-
    movie(Movie),
    good_rated(Movie),
    has_semantic_category(Movie, human_drama).


% ------------------------------------------------------------
% 11. Explanation helper rules
% ------------------------------------------------------------

explain_category(Movie, Category, Reason) :-
    genre(Movie, Genre),
    isa_transitive(Genre, Category),
    title(Movie, Title),
    format(atom(Reason), '~w is connected to ~w through genre ~w.', [Title, Category, Genre]).

explain_category(Movie, Category, Reason) :-
    has_topic(Movie, Topic, Score),
    isa_transitive(Topic, Category),
    title(Movie, Title),
    format(atom(Reason), '~w is connected to ~w through LDA topic ~w with score ~2f.', [Title, Category, Topic, Score]).

explain_category(Movie, Category, Reason) :-
    keyword(Movie, Keyword),
    isa_transitive(Keyword, Category),
    title(Movie, Title),
    format(atom(Reason), '~w is connected to ~w through keyword ~w.', [Title, Category, Keyword]).
"""

with open(ONTOLOGY_OUTPUT, "w", encoding="utf-8") as f:
    f.write(ontology_text)

print(f"Saved: {ONTOLOGY_OUTPUT}")

Saved: /content/CineLog/ontology_rules.pl


In [15]:
with open("/content/CineLog/ontology_rules.pl", "r", encoding="utf-8") as f:
    for i in range(50):
        print(f.readline().rstrip())


% ============================================================
% CineLog Ontology Rules
% STEP2: Multi-label Semantic Net
% ============================================================

% ------------------------------------------------------------
% 1. Genre-level semantic hierarchy
% ------------------------------------------------------------
% Genre is treated as a surface-level category.
% A movie can still belong to other semantic categories through topics/keywords.

isa(action, action_genre).
isa(adventure, action_genre).

isa(comedy, light_genre).
isa(romance, light_genre).
isa(family, light_genre).
isa(animation, light_genre).

isa(drama, narrative_genre).
isa(history, narrative_genre).
isa(documentary, narrative_genre).

isa(crime, dark_genre).
isa(thriller, dark_genre).
isa(horror, dark_genre).
isa(mystery, dark_genre).

isa(science_fiction, speculative_genre).
isa(fantasy, speculative_genre).

isa(war, conflict_genre).
isa(western, conflict_genre).

% Higher-level genre ab

In [16]:
from pyswip import Prolog

prolog = Prolog()
prolog.consult("/content/CineLog/movie_facts.pl")
prolog.consult("/content/CineLog/ontology_rules.pl")

print("movie_facts.pl and ontology_rules.pl loaded successfully.")

movie_facts.pl and ontology_rules.pl loaded successfully.


Semantic Net 작동 테스트

In [17]:
list(prolog.query("isa_transitive(science_fiction, speculative_story)"))

[{}]

In [18]:
list(prolog.query("isa_transitive(action, entertainment)"))

[{}]

In [19]:
list(prolog.query("has_semantic_category(m19995, speculative_story)"))

[{}, {}, {}, {}]

In [20]:
list(prolog.query("has_semantic_category(M, speculative_story), title(M, T)", maxresult=10))

[{'M': 'm19995', 'T': 'Avatar'},
 {'M': 'm19995', 'T': 'Avatar'},
 {'M': 'm285', 'T': "Pirates of the Caribbean: At World's End"},
 {'M': 'm49529', 'T': 'John Carter'},
 {'M': 'm559', 'T': 'Spider-Man 3'},
 {'M': 'm99861', 'T': 'Avengers: Age of Ultron'},
 {'M': 'm767', 'T': 'Harry Potter and the Half-Blood Prince'},
 {'M': 'm209112', 'T': 'Batman v Superman: Dawn of Justice'},
 {'M': 'm1452', 'T': 'Superman Returns'},
 {'M': 'm1452', 'T': 'Superman Returns'}]

In [21]:
list(prolog.query("has_semantic_category(M, dark_story), title(M, T)", maxresult=10))

[{'M': 'm206647', 'T': 'Spectre'},
 {'M': 'm206647', 'T': 'Spectre'},
 {'M': 'm49026', 'T': 'The Dark Knight Rises'},
 {'M': 'm49026', 'T': 'The Dark Knight Rises'},
 {'M': 'm49026', 'T': 'The Dark Knight Rises'},
 {'M': 'm10764', 'T': 'Quantum of Solace'},
 {'M': 'm10764', 'T': 'Quantum of Solace'},
 {'M': 'm10764', 'T': 'Quantum of Solace'},
 {'M': 'm597', 'T': 'Titanic'},
 {'M': 'm44833', 'T': 'Battleship'}]

In [22]:
def show_query(query, limit=10):
    rows = list(prolog.query(query, maxresult=limit))

    if not rows:
        print("No results.")
        return

    for i, row in enumerate(rows, 1):
        print(f"[{i}]")
        for k, v in row.items():
            print(f"  {k}: {v}")
        print()

show_query("has_semantic_category(M, dark_story), title(M, T)", limit=5)

[1]
  M: m206647
  T: Spectre

[2]
  M: m206647
  T: Spectre

[3]
  M: m49026
  T: The Dark Knight Rises

[4]
  M: m49026
  T: The Dark Knight Rises

[5]
  M: m49026
  T: The Dark Knight Rises



In [23]:
show_query("recommend_dark_action(M), title(M, T), rating(M, R)", limit=10)
show_query("recommend_philosophical_sf(M), title(M, T), rating(M, R)", limit=10)
show_query("recommend_light_fun(M), title(M, T), rating(M, R)", limit=10)
show_query("recommend_emotional_movie(M), title(M, T), rating(M, R)", limit=10)

[1]
  M: m49026
  T: The Dark Knight Rises
  R: 7.6

[2]
  M: m49026
  T: The Dark Knight Rises
  R: 7.6

[3]
  M: m49026
  T: The Dark Knight Rises
  R: 7.6

[4]
  M: m49026
  T: The Dark Knight Rises
  R: 7.6

[5]
  M: m767
  T: Harry Potter and the Half-Blood Prince
  R: 7.4

[6]
  M: m58
  T: Pirates of the Caribbean: Dead Man's Chest
  R: 7.0

[7]
  M: m58
  T: Pirates of the Caribbean: Dead Man's Chest
  R: 7.0

[8]
  M: m122917
  T: The Hobbit: The Battle of the Five Armies
  R: 7.1

[9]
  M: m122917
  T: The Hobbit: The Battle of the Five Armies
  R: 7.1

[10]
  M: m57158
  T: The Hobbit: The Desolation of Smaug
  R: 7.6

[1]
  M: m19995
  T: Avatar
  R: 7.2

[2]
  M: m19995
  T: Avatar
  R: 7.2

[3]
  M: m19995
  T: Avatar
  R: 7.2

[4]
  M: m19995
  T: Avatar
  R: 7.2

[5]
  M: m767
  T: Harry Potter and the Half-Blood Prince
  R: 7.4

[6]
  M: m58
  T: Pirates of the Caribbean: Dead Man's Chest
  R: 7.0

[7]
  M: m58
  T: Pirates of the Caribbean: Dead Man's Chest
  R: 7.0



In [24]:
show_query("explain_category(m19995, speculative_story, Reason)", limit=10)

[1]
  Reason: Avatar is connected to speculative_story through genre fantasy.

[2]
  Reason: Avatar is connected to speculative_story through genre science_fiction.

[3]
  Reason: Avatar is connected to speculative_story through LDA topic sf_blockbuster_dystopia with score 0.53.

[4]
  Reason: Avatar is connected to speculative_story through keyword alien.



In [25]:
show_query("explain_category(m19995, human_drama, Reason)", limit=10)

[1]
  Reason: Avatar is connected to human_drama through LDA topic human_journey_true_story with score 0.26.



# DCG 기반 자연어 질의 분석

In [54]:
from pathlib import Path

BASE_DIR = Path("/content/CineLog")
QUERY_RULES_OUTPUT = BASE_DIR / "query_rules.pl"

query_rules_text = r"""
% ============================================================
% CineLog Query Rules
% STEP3: DCG Parser + Lexicon + Skip Rule
% ============================================================

% ------------------------------------------------------------
% 1. Main parser
% ------------------------------------------------------------
% parse_query(+Tokens, -Condition)
% Condition = condition(MinYear, MinRating, Genre, Category, Topic, MaxRuntime, SortMode)

parse_query(Tokens, Condition) :-
    phrase(query_items(Items), Tokens),
    build_condition(Items, Condition).


% ------------------------------------------------------------
% 2. DCG token scanner
% ------------------------------------------------------------
% Each token is mapped into a logical item through lex_item/2.
% Skip words are removed by skip_word/1.

query_items([Item|Rest]) -->
    [Token],
    { lex_item(Token, Item) },
    query_items(Rest).

query_items(Rest) -->
    [Token],
    { skip_word(Token) },
    query_items(Rest).

query_items([]) --> [].


% ------------------------------------------------------------
% 3. Lexicon: time expressions
% ------------------------------------------------------------

lex_item('요즘', year(2015)).
lex_item('최근', year(2015)).
lex_item('최신', year(2018)).
lex_item('옛날', year(0)).
lex_item('고전', year(0)).


% ------------------------------------------------------------
% 4. Lexicon: rating / popularity expressions
% ------------------------------------------------------------
% Ambiguous terms can produce multiple interpretations.
% For example, '재밌는' can mean high rating, popularity, or entertainment.

lex_item('재밌는', rating(8.0)).
lex_item('재밌는', sort(rating)).
lex_item('재밌는', category(entertainment)).

lex_item('명작', rating(8.0)).
lex_item('평점높은', rating(8.0)).
lex_item('평점', rating(7.0)).
lex_item('괜찮은', rating(7.0)).
lex_item('인기있는', sort(popularity)).
lex_item('유명한', sort(popularity)).


% ------------------------------------------------------------
% 5. Lexicon: runtime expressions
% ------------------------------------------------------------

lex_item('짧은', max_runtime(100)).
lex_item('가볍게', max_runtime(110)).
lex_item('긴', min_runtime(140)).


% ------------------------------------------------------------
% 6. Lexicon: genre expressions
% ------------------------------------------------------------

lex_item('액션', genre(action)).
lex_item('모험', genre(adventure)).
lex_item('코미디', genre(comedy)).
lex_item('드라마', genre(drama)).
lex_item('로맨스', genre(romance)).
lex_item('범죄', genre(crime)).
lex_item('스릴러', genre(thriller)).
lex_item('공포', genre(horror)).
lex_item('호러', genre(horror)).
lex_item('미스터리', genre(mystery)).
lex_item('SF', genre(science_fiction)).
lex_item('sf', genre(science_fiction)).
lex_item('판타지', genre(fantasy)).
lex_item('가족', genre(family)).
lex_item('애니메이션', genre(animation)).
lex_item('전쟁', genre(war)).


% ------------------------------------------------------------
% 7. Lexicon: semantic category expressions
% ------------------------------------------------------------
% These categories are connected to ontology_rules.pl.

lex_item('어두운', category(dark_story)).
lex_item('다크한', category(dark_story)).
lex_item('복수', category(dark_story)).
lex_item('잔혹한', category(dark_story)).

lex_item('여운', category(human_drama)).
lex_item('감동', category(human_drama)).
lex_item('휴먼', category(human_drama)).
lex_item('가족적인', category(human_drama)).

lex_item('철학적인', category(speculative_story)).
lex_item('우주', category(speculative_story)).
lex_item('미래', category(speculative_story)).
lex_item('디스토피아', category(speculative_story)).

lex_item('가벼운', category(entertainment)).
lex_item('웃긴', category(entertainment)).
lex_item('킬링타임', category(entertainment)).


% ------------------------------------------------------------
% 8. Lexicon: LDA topic expressions
% ------------------------------------------------------------
% Topic names are based on Topic_Dictionary.docx.

lex_item('우주', topic(sf_blockbuster_dystopia)).
lex_item('외계인', topic(sf_blockbuster_dystopia)).
lex_item('디스토피아', topic(sf_blockbuster_dystopia)).

lex_item('범죄수사', topic(spy_investigation_action)).
lex_item('첩보', topic(spy_investigation_action)).
lex_item('수사', topic(spy_investigation_action)).

lex_item('복수극', topic(crime_thriller_revenge)).
lex_item('살인', topic(crime_thriller_revenge)).

lex_item('청춘', topic(teen_youth_indie)).
lex_item('하이틴', topic(teen_youth_indie)).

lex_item('전쟁', topic(war_epic_period)).
lex_item('시대극', topic(war_epic_period)).

lex_item('오컬트', topic(mystery_occult_adventure)).
lex_item('마녀', topic(mystery_occult_adventure)).

% ------------------------------------------------------------
% 8-2. Advanced Lexicon: ambiguous preference expressions
% ------------------------------------------------------------

% 재미/만족도
lex_item('재밌는', rating(8.0)).
lex_item('재밌는', sort(rating)).
lex_item('재밌는', category(entertainment)).
lex_item('재밌는', genre(comedy)).
lex_item('재밌는', genre(action)).

lex_item('재미있는', rating(8.0)).
lex_item('재미있는', category(entertainment)).
lex_item('볼만한', rating(7.0)).
lex_item('괜찮은', rating(7.0)).
lex_item('명작', rating(8.0)).
lex_item('명작', sort(rating)).
lex_item('수작', rating(7.5)).

% 인기/대중성
lex_item('인기있는', sort(popularity)).
lex_item('인기많은', sort(popularity)).
lex_item('유명한', sort(popularity)).
lex_item('대중적인', sort(popularity)).
lex_item('흥행한', sort(popularity)).
lex_item('블록버스터', sort(popularity)).
lex_item('블록버스터', category(entertainment)).

% 가벼움/킬링타임
lex_item('가벼운', category(entertainment)).
lex_item('가볍게', category(entertainment)).
lex_item('가볍게', max_runtime(110)).
lex_item('킬링타임', category(entertainment)).
lex_item('편하게', category(entertainment)).
lex_item('웃긴', genre(comedy)).
lex_item('웃긴', category(entertainment)).

% 감정적 여운
lex_item('여운', category(human_drama)).
lex_item('여운', category(speculative_story)).
lex_item('여운남는', category(human_drama)).
lex_item('감동', category(human_drama)).
lex_item('감동적인', category(human_drama)).
lex_item('따뜻한', category(human_drama)).
lex_item('휴먼', category(human_drama)).
lex_item('가족적인', category(human_drama)).

% 어두운 분위기
lex_item('어두운', category(dark_story)).
lex_item('다크한', category(dark_story)).
lex_item('무거운', category(dark_story)).
lex_item('잔혹한', category(dark_story)).
lex_item('긴장감있는', category(dark_story)).
lex_item('스릴있는', genre(thriller)).
lex_item('스릴있는', category(dark_story)).

% 철학/SF적 분위기
lex_item('철학적인', category(speculative_story)).
lex_item('생각할거리', category(speculative_story)).
lex_item('미래적인', category(speculative_story)).
lex_item('우주', category(speculative_story)).
lex_item('디스토피아', category(speculative_story)).

% ------------------------------------------------------------
% 9. Skip Rule: logical noise filtering
% ------------------------------------------------------------

skip_word('영화').
skip_word('추천').
skip_word('추천해줘').
skip_word('찾아줘').
skip_word('골라줘').
skip_word('보고싶어').
skip_word('보고').
skip_word('싶어').
skip_word('좀').
skip_word('진짜').
skip_word('완전').
skip_word('중에').
skip_word('중에서').
skip_word('거').
skip_word('것').
skip_word('하나').
skip_word('해줘').
skip_word('있어').
skip_word('있는').
skip_word('으로').
skip_word('로').
skip_word('을').
skip_word('를').
skip_word('이').
skip_word('가').
skip_word('은').
skip_word('는').
skip_word('남는').
skip_word('느낌').
skip_word('스타일').
skip_word('분위기').
skip_word('같은').
skip_word('비슷한').
skip_word('추천해줄래').


% ------------------------------------------------------------
% 10. Build condition from parsed items
% ------------------------------------------------------------

build_condition(Items, condition(MinYear, MinRating, Genre, Category, Topic, MaxRuntime, SortMode)) :-
    choose_year(Items, MinYear),
    choose_rating(Items, MinRating),
    choose_genre(Items, Genre),
    choose_category(Items, Category),
    choose_topic(Items, Topic),
    choose_max_runtime(Items, MaxRuntime),
    choose_sort(Items, SortMode).


choose_year(Items, Year) :-
    member(year(Year), Items), !.
choose_year(_, 0).

choose_rating(Items, Rating) :-
    member(rating(Rating), Items), !.
choose_rating(_, 0.0).

choose_genre(Items, Genre) :-
    member(genre(Genre), Items), !.
choose_genre(_, any).

choose_category(Items, Category) :-
    member(category(Category), Items), !.
choose_category(_, any).

choose_topic(Items, Topic) :-
    member(topic(Topic), Items), !.
choose_topic(_, any).

choose_max_runtime(Items, MaxRuntime) :-
    member(max_runtime(MaxRuntime), Items), !.
choose_max_runtime(_, 999).

choose_sort(Items, SortMode) :-
    member(sort(SortMode), Items), !.
choose_sort(_, default).


% ------------------------------------------------------------
% 11. Condition execution helper
% ------------------------------------------------------------
% These rules will be used in STEP4, but we define them here
% so that the parsed condition can be tested immediately.

satisfy_genre(_, any).
satisfy_genre(Movie, Genre) :-
    Genre \= any,
    is_genre(Movie, Genre).

satisfy_category(_, any).
satisfy_category(Movie, Category) :-
    Category \= any,
    has_semantic_category(Movie, Category).

satisfy_topic(_, any).
satisfy_topic(Movie, Topic) :-
    Topic \= any,
    is_topic(Movie, Topic).

satisfy_runtime(_, 999).
satisfy_runtime(Movie, MaxRuntime) :-
    MaxRuntime \= 999,
    runtime(Movie, T),
    T =< MaxRuntime.


recommend_from_condition(Movie, condition(MinYear, MinRating, Genre, Category, Topic, MaxRuntime, _SortMode)) :-
    movie(Movie),

    % Pruning 1: rating condition
    rating(Movie, R),
    R >= MinRating,

    % Pruning 2: release year condition
    release_year(Movie, Y),
    Y >= MinYear,

    % Pruning 3: runtime condition
    satisfy_runtime(Movie, MaxRuntime),

    % Pruning 4: genre / category / topic conditions
    satisfy_genre(Movie, Genre),
    satisfy_category(Movie, Category),
    satisfy_topic(Movie, Topic).
"""

with open(QUERY_RULES_OUTPUT, "w", encoding="utf-8") as f:
    f.write(query_rules_text)

print(f"Saved: {QUERY_RULES_OUTPUT}")

Saved: /content/CineLog/query_rules.pl


In [55]:
with open("/content/CineLog/query_rules.pl", "r", encoding="utf-8") as f:
    for i in range(40):
        print(f.readline().rstrip())


% ============================================================
% CineLog Query Rules
% STEP3: DCG Parser + Lexicon + Skip Rule
% ============================================================

% ------------------------------------------------------------
% 1. Main parser
% ------------------------------------------------------------
% parse_query(+Tokens, -Condition)
% Condition = condition(MinYear, MinRating, Genre, Category, Topic, MaxRuntime, SortMode)

parse_query(Tokens, Condition) :-
    phrase(query_items(Items), Tokens),
    build_condition(Items, Condition).


% ------------------------------------------------------------
% 2. DCG token scanner
% ------------------------------------------------------------
% Each token is mapped into a logical item through lex_item/2.
% Skip words are removed by skip_word/1.

query_items([Item|Rest]) -->
    [Token],
    { lex_item(Token, Item) },
    query_items(Rest).

query_items(Rest) -->
    [Token],
    { skip_word(Token) },
    query_it

In [56]:
from pyswip import Prolog

prolog = Prolog()
prolog.consult("/content/CineLog/movie_facts.pl")
prolog.consult("/content/CineLog/ontology_rules.pl")
prolog.consult("/content/CineLog/query_rules.pl")

print("All Prolog files loaded successfully.")

All Prolog files loaded successfully.


In [57]:
list(prolog.query(
    "parse_query(['요즘','재밌는','액션','영화','추천해줘'], Condition)",
    maxresult=10
))

[{'Condition': 'condition(2015, 8.0, action, any, any, 999, default)'},
 {'Condition': 'condition(2015, 0.0, action, any, any, 999, rating)'},
 {'Condition': 'condition(2015, 0.0, action, entertainment, any, 999, default)'},
 {'Condition': 'condition(2015, 8.0, action, any, any, 999, default)'},
 {'Condition': 'condition(2015, 0.0, action, any, any, 999, rating)'},
 {'Condition': 'condition(2015, 0.0, action, entertainment, any, 999, default)'},
 {'Condition': 'condition(2015, 0.0, comedy, any, any, 999, default)'},
 {'Condition': 'condition(2015, 0.0, action, any, any, 999, default)'}]

In [58]:
import re

def normalize_query(text):
    """
    한국어 질의에서 자주 붙어 나오는 표현을 STEP3 lexicon에 맞게 정규화.
    """
    text = text.strip()

    replacements = {
        "평점 높은": "평점높은",
        "인기 있는": "인기있는",
        "보고 싶어": "보고싶어",
        "범죄 수사": "범죄수사",
        "복수 극": "복수극",
        "킬링 타임": "킬링타임",
        "SF": "SF",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r"[^\w가-힣\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize_user_query(text):
    normalized = normalize_query(text)
    return normalized.split()


def to_prolog_list(tokens):
    """
    Python token list를 Prolog atom list 문자열로 변환.
    한국어 atom은 작은따옴표로 감싸는 것이 안전함.
    """
    escaped = [t.replace("\\", "\\\\").replace("'", "\\'") for t in tokens]
    return "[" + ",".join([f"'{t}'" for t in escaped]) + "]"


tokens = tokenize_user_query("요즘 재밌는 액션 영화 추천해줘")
print(tokens)
print(to_prolog_list(tokens))

['요즘', '재밌는', '액션', '영화', '추천해줘']
['요즘','재밌는','액션','영화','추천해줘']


In [59]:
def parse_natural_language_query(user_text, max_parses=10):
    tokens = tokenize_user_query(user_text)
    prolog_tokens = to_prolog_list(tokens)

    query = f"parse_query({prolog_tokens}, Condition)"
    rows = list(prolog.query(query, maxresult=max_parses))

    print("[Input]")
    print(user_text)
    print("\n[Tokens]")
    print(tokens)
    print("\n[Parsed Conditions]")

    if not rows:
        print("No parse result.")
        return []

    for i, row in enumerate(rows, 1):
        print(f"{i}. {row['Condition']}")

    return rows


parse_natural_language_query("요즘 재밌는 액션 영화 추천해줘")

[Input]
요즘 재밌는 액션 영화 추천해줘

[Tokens]
['요즘', '재밌는', '액션', '영화', '추천해줘']

[Parsed Conditions]
1. condition(2015, 8.0, action, any, any, 999, default)
2. condition(2015, 0.0, action, any, any, 999, rating)
3. condition(2015, 0.0, action, entertainment, any, 999, default)
4. condition(2015, 8.0, action, any, any, 999, default)
5. condition(2015, 0.0, action, any, any, 999, rating)
6. condition(2015, 0.0, action, entertainment, any, 999, default)
7. condition(2015, 0.0, comedy, any, any, 999, default)
8. condition(2015, 0.0, action, any, any, 999, default)


[{'Condition': 'condition(2015, 8.0, action, any, any, 999, default)'},
 {'Condition': 'condition(2015, 0.0, action, any, any, 999, rating)'},
 {'Condition': 'condition(2015, 0.0, action, entertainment, any, 999, default)'},
 {'Condition': 'condition(2015, 8.0, action, any, any, 999, default)'},
 {'Condition': 'condition(2015, 0.0, action, any, any, 999, rating)'},
 {'Condition': 'condition(2015, 0.0, action, entertainment, any, 999, default)'},
 {'Condition': 'condition(2015, 0.0, comedy, any, any, 999, default)'},
 {'Condition': 'condition(2015, 0.0, action, any, any, 999, default)'}]

In [60]:
parse_natural_language_query("요즘 재밌는 액션 영화 추천해줘")

[Input]
요즘 재밌는 액션 영화 추천해줘

[Tokens]
['요즘', '재밌는', '액션', '영화', '추천해줘']

[Parsed Conditions]
1. condition(2015, 8.0, action, any, any, 999, default)
2. condition(2015, 0.0, action, any, any, 999, rating)
3. condition(2015, 0.0, action, entertainment, any, 999, default)
4. condition(2015, 8.0, action, any, any, 999, default)
5. condition(2015, 0.0, action, any, any, 999, rating)
6. condition(2015, 0.0, action, entertainment, any, 999, default)
7. condition(2015, 0.0, comedy, any, any, 999, default)
8. condition(2015, 0.0, action, any, any, 999, default)


[{'Condition': 'condition(2015, 8.0, action, any, any, 999, default)'},
 {'Condition': 'condition(2015, 0.0, action, any, any, 999, rating)'},
 {'Condition': 'condition(2015, 0.0, action, entertainment, any, 999, default)'},
 {'Condition': 'condition(2015, 8.0, action, any, any, 999, default)'},
 {'Condition': 'condition(2015, 0.0, action, any, any, 999, rating)'},
 {'Condition': 'condition(2015, 0.0, action, entertainment, any, 999, default)'},
 {'Condition': 'condition(2015, 0.0, comedy, any, any, 999, default)'},
 {'Condition': 'condition(2015, 0.0, action, any, any, 999, default)'}]

In [61]:
parse_natural_language_query("평점 높은 SF 영화 찾아줘")

[Input]
평점 높은 SF 영화 찾아줘

[Tokens]
['평점높은', 'SF', '영화', '찾아줘']

[Parsed Conditions]
1. condition(0, 8.0, science_fiction, any, any, 999, default)


[{'Condition': 'condition(0, 8.0, science_fiction, any, any, 999, default)'}]

In [62]:
parse_natural_language_query("어두운 범죄 스릴러 영화 추천해줘")

[Input]
어두운 범죄 스릴러 영화 추천해줘

[Tokens]
['어두운', '범죄', '스릴러', '영화', '추천해줘']

[Parsed Conditions]
1. condition(0, 0.0, crime, dark_story, any, 999, default)
2. condition(0, 0.0, crime, dark_story, any, 999, default)


[{'Condition': 'condition(0, 0.0, crime, dark_story, any, 999, default)'},
 {'Condition': 'condition(0, 0.0, crime, dark_story, any, 999, default)'}]

In [63]:
parse_natural_language_query("여운 남는 감동 영화 추천해줘")

[Input]
여운 남는 감동 영화 추천해줘

[Tokens]
['여운', '남는', '감동', '영화', '추천해줘']

[Parsed Conditions]
1. condition(0, 0.0, any, human_drama, any, 999, default)
2. condition(0, 0.0, any, human_drama, any, 999, default)
3. condition(0, 0.0, any, human_drama, any, 999, default)
4. condition(0, 0.0, any, human_drama, any, 999, default)
5. condition(0, 0.0, any, speculative_story, any, 999, default)
6. condition(0, 0.0, any, speculative_story, any, 999, default)


[{'Condition': 'condition(0, 0.0, any, human_drama, any, 999, default)'},
 {'Condition': 'condition(0, 0.0, any, human_drama, any, 999, default)'},
 {'Condition': 'condition(0, 0.0, any, human_drama, any, 999, default)'},
 {'Condition': 'condition(0, 0.0, any, human_drama, any, 999, default)'},
 {'Condition': 'condition(0, 0.0, any, speculative_story, any, 999, default)'},
 {'Condition': 'condition(0, 0.0, any, speculative_story, any, 999, default)'}]

In [64]:
parse_natural_language_query("짧은 코미디 영화 추천해줘")

[Input]
짧은 코미디 영화 추천해줘

[Tokens]
['짧은', '코미디', '영화', '추천해줘']

[Parsed Conditions]
1. condition(0, 0.0, comedy, any, any, 100, default)


[{'Condition': 'condition(0, 0.0, comedy, any, any, 100, default)'}]

In [65]:
def run_condition_test(user_text, max_parses=3, limit=10):
    tokens = tokenize_user_query(user_text)
    prolog_tokens = to_prolog_list(tokens)

    parse_query_str = f"parse_query({prolog_tokens}, Condition)"
    parsed = list(prolog.query(parse_query_str, maxresult=max_parses))

    if not parsed:
        print("No parse result.")
        return

    print("[Input]", user_text)
    print("[Tokens]", tokens)
    print()

    for idx, row in enumerate(parsed, 1):
        condition = row["Condition"]
        print(f"=== Parse {idx}: {condition} ===")

        recommend_query = (
            f"recommend_from_condition(M, {condition}), "
            f"title(M, T), rating(M, R)"
        )

        results = list(prolog.query(recommend_query, maxresult=limit))

        if not results:
            print("No recommendation result.\n")
            continue

        for i, r in enumerate(results, 1):
            print(f"{i}. {r['T']} | rating: {r['R']} | id: {r['M']}")
        print()


run_condition_test("요즘 재밌는 액션 영화 추천해줘", max_parses=3, limit=10)

[Input] 요즘 재밌는 액션 영화 추천해줘
[Tokens] ['요즘', '재밌는', '액션', '영화', '추천해줘']

=== Parse 1: condition(2015, 8.0, action, any, any, 999, default) ===
No recommendation result.

=== Parse 2: condition(2015, 0.0, action, any, any, 999, rating) ===
1. Spectre | rating: 6.3 | id: m206647
2. Avengers: Age of Ultron | rating: 7.3 | id: m99861
3. Batman v Superman: Dawn of Justice | rating: 5.7 | id: m209112
4. Captain America: Civil War | rating: 7.1 | id: m271110
5. Jurassic World | rating: 6.5 | id: m135397
6. Furious 7 | rating: 7.3 | id: m168259
7. Star Trek Beyond | rating: 6.6 | id: m188927
8. Jupiter Ascending | rating: 5.2 | id: m76757
9. The Legend of Tarzan | rating: 5.5 | id: m258489
10. Suicide Squad | rating: 5.9 | id: m297761

=== Parse 3: condition(2015, 0.0, action, entertainment, any, 999, default) ===
1. Spectre | rating: 6.3 | id: m206647
2. Spectre | rating: 6.3 | id: m206647
3. Avengers: Age of Ultron | rating: 7.3 | id: m99861
4. Avengers: Age of Ultron | rating: 7.3 | id: m99861

“재밌는”이라는 표현은 사용자마다 의미가 다를 수 있으므로, Lexicon에서 이를 평점 기반, 인기도 기반, 엔터테인먼트성 기반의 복수 해석 후보로 확장하였다. 이는 사용자가 명시적 니즈를 가지고 있지 않다는 문제를 해결하기 위한 Preference Clarification 구조이다.

In [66]:
parse_natural_language_query("재밌는 영화 추천해줘", max_parses=10)

[Input]
재밌는 영화 추천해줘

[Tokens]
['재밌는', '영화', '추천해줘']

[Parsed Conditions]
1. condition(0, 8.0, any, any, any, 999, default)
2. condition(0, 0.0, any, any, any, 999, rating)
3. condition(0, 0.0, any, entertainment, any, 999, default)
4. condition(0, 8.0, any, any, any, 999, default)
5. condition(0, 0.0, any, any, any, 999, rating)
6. condition(0, 0.0, any, entertainment, any, 999, default)
7. condition(0, 0.0, comedy, any, any, 999, default)
8. condition(0, 0.0, action, any, any, 999, default)


[{'Condition': 'condition(0, 8.0, any, any, any, 999, default)'},
 {'Condition': 'condition(0, 0.0, any, any, any, 999, rating)'},
 {'Condition': 'condition(0, 0.0, any, entertainment, any, 999, default)'},
 {'Condition': 'condition(0, 8.0, any, any, any, 999, default)'},
 {'Condition': 'condition(0, 0.0, any, any, any, 999, rating)'},
 {'Condition': 'condition(0, 0.0, any, entertainment, any, 999, default)'},
 {'Condition': 'condition(0, 0.0, comedy, any, any, 999, default)'},
 {'Condition': 'condition(0, 0.0, action, any, any, 999, default)'}]

# Rule-based Inference

In [67]:
from pyswip import Prolog

prolog = Prolog()
prolog.consult("/content/CineLog/movie_facts.pl")
prolog.consult("/content/CineLog/ontology_rules.pl")
prolog.consult("/content/CineLog/query_rules.pl")

print("All Prolog files loaded successfully.")

All Prolog files loaded successfully.


In [68]:
import re

def normalize_query(text):
    text = text.strip()

    replacements = {
        "평점 높은": "평점높은",
        "평점 좋은": "평점좋은",
        "인기 있는": "인기있는",
        "인기 많은": "인기많은",
        "보고 싶어": "보고싶어",
        "범죄 수사": "범죄수사",
        "복수 극": "복수극",
        "킬링 타임": "킬링타임",
        "여운 남는": "여운남는",
        "생각할 거리": "생각할거리",
        "긴장감 있는": "긴장감있는",
        "스릴 있는": "스릴있는",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r"[^\w가-힣\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize_user_query(text):
    normalized = normalize_query(text)
    return normalized.split()


def to_prolog_list(tokens):
    escaped = [t.replace("\\", "\\\\").replace("'", "\\'") for t in tokens]
    return "[" + ",".join([f"'{t}'" for t in escaped]) + "]"

In [69]:
def parse_conditions(user_text, max_parses=20):
    tokens = tokenize_user_query(user_text)
    prolog_tokens = to_prolog_list(tokens)

    query = f"parse_query({prolog_tokens}, Condition)"
    rows = list(prolog.query(query, maxresult=max_parses))

    conditions = []
    seen = set()

    for row in rows:
        condition = str(row["Condition"])
        if condition not in seen:
            seen.add(condition)
            conditions.append(condition)

    return {
        "input": user_text,
        "tokens": tokens,
        "conditions": conditions
    }


parsed = parse_conditions("재밌는 영화 추천해줘")
parsed

{'input': '재밌는 영화 추천해줘',
 'tokens': ['재밌는', '영화', '추천해줘'],
 'conditions': ['condition(0, 8.0, any, any, any, 999, default)',
  'condition(0, 0.0, any, any, any, 999, rating)',
  'condition(0, 0.0, any, entertainment, any, 999, default)',
  'condition(0, 0.0, comedy, any, any, 999, default)',
  'condition(0, 0.0, action, any, any, 999, default)']}

In [70]:
def get_candidates_from_condition(condition, max_results=300):
    query = (
        f"recommend_from_condition(M, {condition}), "
        f"title(M, T), "
        f"rating(M, R), "
        f"popularity(M, P), "
        f"release_year(M, Y)"
    )

    rows = list(prolog.query(query, maxresult=max_results))

    candidates = []
    for row in rows:
        candidates.append({
            "movie": str(row["M"]),
            "title": str(row["T"]),
            "rating": float(row["R"]),
            "popularity": float(row["P"]),
            "release_year": int(row["Y"]),
            "condition": condition
        })

    return candidates


candidates = get_candidates_from_condition(
    "condition(0, 8.0, any, any, any, 999, default)",
    max_results=10
)

candidates[:3]

[{'movie': 'm155',
  'title': 'The Dark Knight',
  'rating': 8.2,
  'popularity': 187.322927,
  'release_year': 2008,
  'condition': 'condition(0, 8.0, any, any, any, 999, default)'},
 {'movie': 'm150540',
  'title': 'Inside Out',
  'rating': 8.0,
  'popularity': 128.655964,
  'release_year': 2015,
  'condition': 'condition(0, 8.0, any, any, any, 999, default)'},
 {'movie': 'm157336',
  'title': 'Interstellar',
  'rating': 8.1,
  'popularity': 724.247784,
  'release_year': 2014,
  'condition': 'condition(0, 8.0, any, any, any, 999, default)'}]

In [71]:
def parse_condition_string(condition):
    """
    condition(0, 8.0, any, entertainment, any, 999, default)
    -> dict
    """
    inner = condition.strip()
    inner = inner[len("condition("):-1]
    parts = [p.strip() for p in inner.split(",")]

    return {
        "min_year": int(float(parts[0])),
        "min_rating": float(parts[1]),
        "genre": parts[2],
        "category": parts[3],
        "topic": parts[4],
        "max_runtime": int(float(parts[5])),
        "sort_mode": parts[6]
    }


parse_condition_string("condition(0, 8.0, any, any, any, 999, default)")

{'min_year': 0,
 'min_rating': 8.0,
 'genre': 'any',
 'category': 'any',
 'topic': 'any',
 'max_runtime': 999,
 'sort_mode': 'default'}

In [72]:
def get_topic_score(movie_atom, topic):
    if topic == "any":
        return 0.0

    rows = list(prolog.query(f"has_topic({movie_atom}, {topic}, Score)", maxresult=1))
    if rows:
        return float(rows[0]["Score"])

    # topic이 상위 category인 경우 직접 has_topic은 없을 수 있으므로,
    # 연결된 topic들의 score 중 최대값을 사용
    query = (
        f"has_topic({movie_atom}, Topic, Score), "
        f"isa_transitive(Topic, {topic})"
    )
    rows = list(prolog.query(query, maxresult=20))
    if rows:
        return max(float(r["Score"]) for r in rows)

    return 0.0

여기서는 해당 category로 연결되는 LDA topic 점수가 있으면 그 최대값을 사용하고, 없으면 0.15 정도의 기본 연결 점수 부여

In [73]:
def get_category_strength(movie_atom, category):
    if category == "any":
        return 0.0

    # LDA topic을 통해 category에 연결된 경우 topic score 사용
    query = (
        f"has_topic({movie_atom}, Topic, Score), "
        f"isa_transitive(Topic, {category})"
    )
    rows = list(prolog.query(query, maxresult=20))
    if rows:
        return max(float(r["Score"]) for r in rows)

    # genre나 keyword를 통해 category에 연결되면 기본 점수 부여
    rows = list(prolog.query(f"has_semantic_category({movie_atom}, {category})", maxresult=1))
    if rows:
        return 0.15

    return 0.0

In [74]:
def normalize_rating(rating):
    return max(0.0, min(rating / 10.0, 1.0))


def normalize_popularity(popularity):
    # TMDB popularity는 일부 극단값이 있으므로 완만하게 scaling
    return max(0.0, min(popularity / 100.0, 1.0))


def normalize_recency(year):
    # TMDB 5000 기준에서 1980년 이전은 낮게, 2020에 가까울수록 높게
    if year <= 1980:
        return 0.0
    if year >= 2020:
        return 1.0
    return (year - 1980) / 40.0

In [75]:
def compute_final_score(candidate, condition):
    c = parse_condition_string(condition)

    rating_score = normalize_rating(candidate["rating"])
    popularity_score = normalize_popularity(candidate["popularity"])
    recency_score = normalize_recency(candidate["release_year"])

    topic_score = get_topic_score(candidate["movie"], c["topic"])
    category_score = get_category_strength(candidate["movie"], c["category"])
    semantic_score = max(topic_score, category_score)

    sort_mode = c["sort_mode"]

    if sort_mode == "rating":
        final_score = (
            0.55 * rating_score +
            0.20 * semantic_score +
            0.15 * popularity_score +
            0.10 * recency_score
        )
    elif sort_mode == "popularity":
        final_score = (
            0.30 * rating_score +
            0.20 * semantic_score +
            0.40 * popularity_score +
            0.10 * recency_score
        )
    else:
        final_score = (
            0.40 * rating_score +
            0.25 * semantic_score +
            0.20 * popularity_score +
            0.15 * recency_score
        )

    return {
        "final_score": final_score,
        "rating_score": rating_score,
        "semantic_score": semantic_score,
        "popularity_score": popularity_score,
        "recency_score": recency_score
    }

In [84]:
KOR_LABELS = {
    # Semantic categories
    "entertainment": "오락성/대중성",
    "dark_story": "어두운 서사",
    "human_drama": "휴먼 드라마",
    "speculative_story": "SF·판타지적 상상력",
    "conflict_story": "갈등·전쟁 서사",
    "story_based": "서사 중심",
    "action_genre": "액션 계열",
    "light_genre": "가벼운 장르",
    "narrative_genre": "서사 장르",
    "dark_genre": "어두운 장르",
    "speculative_genre": "SF·판타지 장르",

    # Genres
    "action": "액션",
    "adventure": "모험",
    "comedy": "코미디",
    "romance": "로맨스",
    "family": "가족",
    "animation": "애니메이션",
    "drama": "드라마",
    "history": "역사",
    "documentary": "다큐멘터리",
    "crime": "범죄",
    "thriller": "스릴러",
    "horror": "공포",
    "mystery": "미스터리",
    "science_fiction": "SF",
    "fantasy": "판타지",
    "war": "전쟁",

    # LDA topics
    "family_generational_drama": "가족 및 세대 드라마",
    "war_epic_period": "전쟁·대서사시·시대극",
    "human_journey_true_story": "휴먼 저니·감동 실화",
    "crime_thriller_revenge": "범죄 스릴러·복수극",
    "romantic_comedy_family": "로맨틱 코미디·가정 드라마",
    "teen_youth_indie": "하이틴·청춘·독립영화",
    "mystery_occult_adventure": "미스터리·오컬트·모험",
    "campus_subculture_social": "캠퍼스·서브컬처·사회 고발",
    "sf_blockbuster_dystopia": "SF 블록버스터·디스토피아",
    "spy_investigation_action": "첩보·수사·액션",
}


def ko_label(atom):
    atom = str(atom)
    return KOR_LABELS.get(atom, atom.replace("_", " "))

In [85]:
def get_basic_movie_info(movie_atom):
    info = {}

    for pred, key in [
        ("title", "title"),
        ("rating", "rating"),
        ("popularity", "popularity"),
        ("release_year", "release_year"),
        ("runtime", "runtime"),
    ]:
        rows = list(prolog.query(f"{pred}({movie_atom}, X)", maxresult=1))
        if rows:
            info[key] = rows[0]["X"]

    return info


def get_genres(movie_atom, limit=10):
    rows = list(prolog.query(f"genre({movie_atom}, G)", maxresult=limit))
    return [str(r["G"]) for r in rows]


def get_topics(movie_atom, limit=3):
    rows = list(prolog.query(f"has_topic({movie_atom}, Topic, Score)", maxresult=limit))
    return [(str(r["Topic"]), float(r["Score"])) for r in rows]


def get_category_reasons_ko(movie_atom, category, limit=5):
    if category == "any":
        return []

    reasons = []
    seen = set()

    # 1. 장르를 통한 연결 근거
    genre_query = (
        f"genre({movie_atom}, Genre), "
        f"isa_transitive(Genre, {category})"
    )
    for row in prolog.query(genre_query, maxresult=limit):
        genre = str(row["Genre"])
        reason = (
            f"장르 '{ko_label(genre)}'가 "
            f"'{ko_label(category)}' 범주와 연결됩니다."
        )
        if reason not in seen:
            seen.add(reason)
            reasons.append(reason)

    # 2. LDA 토픽을 통한 연결 근거
    topic_query = (
        f"has_topic({movie_atom}, Topic, Score), "
        f"isa_transitive(Topic, {category})"
    )
    for row in prolog.query(topic_query, maxresult=limit):
        topic = str(row["Topic"])
        score = float(row["Score"])
        reason = (
            f"LDA 토픽 '{ko_label(topic)}' 점수 {score:.2f}를 통해 "
            f"'{ko_label(category)}' 범주와 연결됩니다."
        )
        if reason not in seen:
            seen.add(reason)
            reasons.append(reason)

    # 3. 키워드를 통한 연결 근거
    keyword_query = (
        f"keyword({movie_atom}, Keyword), "
        f"isa_transitive(Keyword, {category})"
    )
    for row in prolog.query(keyword_query, maxresult=limit):
        keyword = str(row["Keyword"])
        reason = (
            f"키워드 '{ko_label(keyword)}'를 통해 "
            f"'{ko_label(category)}' 범주와 연결됩니다."
        )
        if reason not in seen:
            seen.add(reason)
            reasons.append(reason)

    return reasons[:limit]


def generate_explanation(candidate, condition, score_detail):
    c = parse_condition_string(condition)
    movie = candidate["movie"]

    explanations = []

    explanations.append(
        f"평점 {candidate['rating']:.1f}점으로 조건의 최소 평점 {c['min_rating']:.1f} 이상을 만족합니다."
    )

    if c["min_year"] > 0:
        explanations.append(
            f"{candidate['release_year']}년 개봉작으로, {c['min_year']}년 이후 조건을 만족합니다."
        )

    if c["genre"] != "any":
        genres = get_genres(movie)
        genre_text = ", ".join([ko_label(g) for g in genres[:5]])
        explanations.append(
            f"요청한 장르 '{ko_label(c['genre'])}'와 관련되며, 실제 장르는 {genre_text}입니다."
        )

    if c["category"] != "any":
        reasons = get_category_reasons_ko(movie, c["category"])
        if reasons:
            explanations.append(
                f"Semantic Net상 '{ko_label(c['category'])}' 범주와 연결됩니다."
            )
            for reason in reasons[:2]:
                explanations.append(reason)

    if c["topic"] != "any":
        topic_score = get_topic_score(movie, c["topic"])
        explanations.append(
            f"LDA 토픽 '{ko_label(c['topic'])}' 관련 점수는 {topic_score:.2f}입니다."
        )

    topics = get_topics(movie)
    if topics:
        topic_text = ", ".join([f"{ko_label(t)}({s:.2f})" for t, s in topics])
        explanations.append(f"주요 LDA 토픽: {topic_text}")

    explanations.append(
        f"최종 점수는 {score_detail['final_score']:.3f}입니다 "
        f"(평점 {score_detail['rating_score']:.2f}, 의미 {score_detail['semantic_score']:.2f}, "
        f"인기도 {score_detail['popularity_score']:.2f}, 최신성 {score_detail['recency_score']:.2f})."
    )

    return explanations

In [86]:
def recommend_movies(user_text, max_parses=10, candidates_per_condition=300, top_n=10):
    parsed = parse_conditions(user_text, max_parses=max_parses)

    all_candidates = []

    for condition in parsed["conditions"]:
        candidates = get_candidates_from_condition(
            condition,
            max_results=candidates_per_condition
        )

        for cand in candidates:
            score_detail = compute_final_score(cand, condition)

            enriched = {
                **cand,
                "condition": condition,
                **score_detail
            }

            all_candidates.append(enriched)

    # 같은 영화가 여러 condition에서 나올 수 있으므로,
    # 우선은 가장 높은 final_score를 가진 경우만 남김
    best_by_movie = {}

    for cand in all_candidates:
        movie = cand["movie"]
        if movie not in best_by_movie or cand["final_score"] > best_by_movie[movie]["final_score"]:
            best_by_movie[movie] = cand

    ranked = sorted(
        best_by_movie.values(),
        key=lambda x: x["final_score"],
        reverse=True
    )

    top_results = ranked[:top_n]

    for result in top_results:
        result["explanations"] = generate_explanation(
            result,
            result["condition"],
            result
        )

    return {
        "input": parsed["input"],
        "tokens": parsed["tokens"],
        "conditions": parsed["conditions"],
        "results": top_results
    }

In [87]:
def print_recommendation(result):
    print("=" * 80)
    print("[Input]")
    print(result["input"])

    print("\n[Tokens]")
    print(result["tokens"])

    print("\n[Parsed Conditions]")
    for i, condition in enumerate(result["conditions"], 1):
        print(f"{i}. {condition}")

    print("\n[Top Recommendations]")
    if not result["results"]:
        print("No recommendation result.")
        return

    for i, item in enumerate(result["results"], 1):
        print("-" * 80)
        print(f"{i}. {item['title']} ({item['release_year']})")
        print(f"   ID: {item['movie']}")
        print(f"   Rating: {item['rating']:.1f}")
        print(f"   Popularity: {item['popularity']:.2f}")
        print(f"   Final Score: {item['final_score']:.3f}")
        print(f"   Selected Condition: {item['condition']}")
        print("   추천 근거:")
        for exp in item["explanations"]:
            print(f"   - {exp}")
    print("=" * 80)

테스트 실행

In [88]:
result = recommend_movies("재밌는 영화 추천해줘", top_n=5)
print_recommendation(result)

[Input]
재밌는 영화 추천해줘

[Tokens]
['재밌는', '영화', '추천해줘']

[Parsed Conditions]
1. condition(0, 8.0, any, any, any, 999, default)
2. condition(0, 0.0, any, any, any, 999, rating)
3. condition(0, 0.0, any, entertainment, any, 999, default)
4. condition(0, 0.0, comedy, any, any, 999, default)
5. condition(0, 0.0, action, any, any, 999, default)

[Top Recommendations]
--------------------------------------------------------------------------------
1. Interstellar (2014)
   ID: m157336
   Rating: 8.1
   Popularity: 724.25
   Final Score: 0.689
   Selected Condition: condition(0, 0.0, any, entertainment, any, 999, default)
   추천 근거:
   - 평점 8.1점으로 조건의 최소 평점 0.0 이상을 만족합니다.
   - Semantic Net상 '오락성/대중성' 범주와 연결됩니다.
   - 장르 '모험'가 '오락성/대중성' 범주와 연결됩니다.
   - 주요 LDA 토픽: SF 블록버스터·디스토피아(0.62), 휴먼 저니·감동 실화(0.36)
   - 최종 점수는 0.689입니다 (평점 0.81, 의미 0.15, 인기도 1.00, 최신성 0.85).
--------------------------------------------------------------------------------
2. Inside Out (2015)
   ID: m150540
   Rating: 8.0
   Popu